<a href="https://colab.research.google.com/github/droyktton/ICNPG/blob/master/Clases/Clases_aleatorios/ICNPG_aleatorios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Generación de números aleatorios en paralelo**

# curand HOST API

Empecemos con la API del host, que puede ser usada para generar un array random en host o en device

In [ ]:
%%writefile randomarray.cu
/*
 * This program uses the host CURAND API to generate 100
 * pseudorandom floats.
 */
 #include <stdio.h>
 #include <stdlib.h>
 #include <cuda.h>
 #include <curand.h>

 #define CUDA_CALL(x) do { if((x)!=cudaSuccess) { \
     printf("Error at %s:%d\n",__FILE__,__LINE__);\
     return EXIT_FAILURE;}} while(0)
 #define CURAND_CALL(x) do { if((x)!=CURAND_STATUS_SUCCESS) { \
     printf("Error at %s:%d\n",__FILE__,__LINE__);\
     return EXIT_FAILURE;}} while(0)

 int main(int argc, char *argv[])
 {

     size_t n = 100;

     if(argc>1) n=atoi(argv[1]);

     size_t i;
     curandGenerator_t gen;
     float *hostData;

     /* Allocate n floats on host */
     hostData = (float *)calloc(n, sizeof(float));

    #ifndef CPU
    float *devData;

     /* Allocate n floats on device */
     CUDA_CALL(cudaMalloc((void **)&devData, n*sizeof(float)));

     /* Create pseudo-random number generator */
     CURAND_CALL(curandCreateGenerator(&gen,
                 CURAND_RNG_PSEUDO_DEFAULT));

     /* Set seed */
     CURAND_CALL(curandSetPseudoRandomGeneratorSeed(gen,
                 1234ULL));

     /* Generate n floats on device */
     CURAND_CALL(curandGenerateUniform(gen, devData, n));

     /* Copy device memory to host */
     CUDA_CALL(cudaMemcpy(hostData, devData, n * sizeof(float),
         cudaMemcpyDeviceToHost));
    #else

     /* Create pseudo-random number generator */
     CURAND_CALL(curandCreateGeneratorHost(&gen,
        CURAND_RNG_PSEUDO_DEFAULT));

     /* Set seed */
     CURAND_CALL(curandSetPseudoRandomGeneratorSeed(gen,
        1234ULL));

     /* Generate n floats on device */
     CURAND_CALL(curandGenerateUniform(gen, hostData, n));

     #endif

     /* Show result */
     int m=(n>100)?100:n;
     for(i = 0; i < m; i++) {
         printf("%1.4f ", hostData[i]);
     }
     printf("\n");

     /* Cleanup */
     CURAND_CALL(curandDestroyGenerator(gen));

     #ifndef CPU
     CUDA_CALL(cudaFree(devData));
     #endif
     free(hostData);

     return EXIT_SUCCESS;
 }


Writing randomarray.cu


In [ ]:
!nvcc -arch=sm_75 randomarray.cu -o random -lcurand
! ./random

0.1455 0.8202 0.5504 0.2948 0.9147 0.8690 0.3219 0.7829 0.0113 0.2855 0.7816 0.2338 0.6791 0.2824 0.6299 0.1212 0.4333 0.3831 0.5136 0.2987 0.4166 0.0345 0.0494 0.0467 0.6166 0.6480 0.8685 0.4012 0.0631 0.4972 0.6809 0.9350 0.0704 0.0458 0.1324 0.3785 0.6457 0.9930 0.9952 0.7677 0.3217 0.8210 0.2765 0.2691 0.4579 0.1969 0.9555 0.8739 0.7996 0.3810 0.6662 0.3153 0.9428 0.5006 0.3369 0.1490 0.8637 0.6191 0.6820 0.4573 0.9261 0.5650 0.7117 0.8252 0.8755 0.2216 0.2958 0.4046 0.3896 0.7335 0.7301 0.8154 0.0913 0.0866 0.6974 0.1811 0.5834 0.9255 0.9029 0.0413 0.9522 0.5507 0.7237 0.3976 0.7519 0.4398 0.4638 0.6094 0.7358 0.3272 0.6961 0.4893 0.9698 0.0456 0.2025 0.9491 0.1516 0.0424 0.6149 0.5638 


Si definimos el macro CPU en la compilacion podemos generar directamente en el host

In [ ]:
!nvcc -arch=sm_75 randomarray.cu -o random -lcurand -DCPU
! ./random

0.1455 0.8202 0.5504 0.2948 0.9147 0.8690 0.3219 0.7829 0.0113 0.2855 0.7816 0.2338 0.6791 0.2824 0.6299 0.1212 0.4333 0.3831 0.5136 0.2987 0.4166 0.0345 0.0494 0.0467 0.6166 0.6480 0.8685 0.4012 0.0631 0.4972 0.6809 0.9350 0.0704 0.0458 0.1324 0.3785 0.6457 0.9930 0.9952 0.7677 0.3217 0.8210 0.2765 0.2691 0.4579 0.1969 0.9555 0.8739 0.7996 0.3810 0.6662 0.3153 0.9428 0.5006 0.3369 0.1490 0.8637 0.6191 0.6820 0.4573 0.9261 0.5650 0.7117 0.8252 0.8755 0.2216 0.2958 0.4046 0.3896 0.7335 0.7301 0.8154 0.0913 0.0866 0.6974 0.1811 0.5834 0.9255 0.9029 0.0413 0.9522 0.5507 0.7237 0.3976 0.7519 0.4398 0.4638 0.6094 0.7358 0.3272 0.6961 0.4893 0.9698 0.0456 0.2025 0.9491 0.1516 0.0424 0.6149 0.5638 


Como usamos la misma semilla para ambos, tiene sentido tener los mismos numeros.

**Ejercicios:**
* Timming CPU vs GPU.
* Verificar distribución.
* Cambiar distribución.      


## Calcular $\pi$ con numeros aleatorios

Empecemos en C++, usando curand y thrust, pero usando la HOST API.

In [ ]:
%%writefile Pi_curand.cu

 #include <iostream>
 #include <stdlib.h>
 #include <cuda.h>
 #include <curand.h>
 #include <thrust/device_vector.h>
 #include <thrust/count.h>

 #define CUDA_CALL(x) do { if((x)!=cudaSuccess) { \
     printf("Error at %s:%d\n",__FILE__,__LINE__);\
     return EXIT_FAILURE;}} while(0)
 #define CURAND_CALL(x) do { if((x)!=CURAND_STATUS_SUCCESS) { \
     printf("Error at %s:%d\n",__FILE__,__LINE__);\
     return EXIT_FAILURE;}} while(0)


struct dentroCirculoUnidad
{
    __host__ __device__
    bool operator()(thrust::tuple<float,float> tup)
    {
        float x=thrust::get<0>(tup);
        float y=thrust::get<1>(tup);
        return (x*x+y*y)<1;
    }
};

 int main(int argc, char *argv[])
 {
    size_t n = 10000000;
    curandGenerator_t gen;

    thrust::device_vector<float> devData(n);

     /* Create pseudo-random number generator */
     CURAND_CALL(curandCreateGenerator(&gen,
                 CURAND_RNG_PSEUDO_DEFAULT));

     /* Set seed */
     CURAND_CALL(curandSetPseudoRandomGeneratorSeed(gen,
                 1234ULL));

    /* Generate n floats on device */
    float *devData_raw=thrust::raw_pointer_cast(&devData[0]);
    CURAND_CALL(curandGenerateUniform(gen, devData_raw, n));

    // usamos la primera mitad como x, la segunda como y
    int adentro = thrust::count_if(
        thrust::make_zip_iterator((thrust::make_tuple(devData.begin(),devData.begin()+n/2))),
        thrust::make_zip_iterator((thrust::make_tuple(devData.begin()+n/2,devData.begin()+n))),
        dentroCirculoUnidad()
    );

    /*
    dardos adentro = rho pi R^2/4
    dardos total = rho R^2
    4*adentro/total = pi
    */

    // hay n/2 dardos ->
    std::cout << 4.0*adentro*(2.0/n) << std::endl;

     /* Cleanup */
     CURAND_CALL(curandDestroyGenerator(gen));
     return EXIT_SUCCESS;
 }

Writing Pi_curand.cu


In [ ]:
!nvcc -arch=sm_75 Pi_curand.cu -o Pi_curand -lcurand
! ./Pi_curand

3.14127


Ahora hagamoslo con cupy, usando tambien la HOST API, y comparemos con numpy

In [ ]:
import cupy as cp
import numpy as np
from cupyx.time import repeat

def piCPU(n):
    ## en CPU
    data = np.random.uniform(-0.5,0.5,(n,2))

    inside = len(
        np.argwhere(
            np.linalg.norm(data, axis=1) < 0.5
        )
    )
    print(inside/n*4)

def piGPU(n):
    ## en GPU
    dataGPU = cp.random.uniform(-0.5,0.5,(n,2))

    inside = len(
        cp.argwhere(
            cp.linalg.norm(dataGPU, axis=1) < 0.5
        )
    )
    print(inside/n*4)

n=10000000

print(repeat(piCPU, (n,), n_repeat=10))
print(repeat(piGPU, (n,), n_repeat=10))

#piCPU(n)
#piGPU(n)


/usr/local/lib/python3.11/dist-packages/cupyx/time.py:62: UserWarning: cupyx.time.repeat has been moved to cupyx.profiler.benchmark since CuPy v10. Access through cupyx.time is deprecated and will be removed in the future.
  _warnings.warn(


3.1417104
3.1413364
3.1411628
3.1413636
3.1421972
3.141748
3.1415708
3.141974
3.1418716
3.1421956
3.1432732
3.1407524
3.142126
3.1414776
3.1418608
3.1417092
3.1426416
3.1411164
3.1412772
3.142442
piCPU               :    CPU: 505606.699 us   +/- 4086.073 (min: 498985.292 / max: 511794.560) us     GPU-0: 505672.116 us   +/- 4086.312 (min: 499051.270 / max: 511858.887) us
3.1422092
3.141676
3.1411188
3.1421916
3.1412204
3.1414732
3.1410132
3.1411432
3.1411296
3.1417052
3.140528
3.1415224
3.1410112
3.1413236
3.1411032
3.1424276
3.1404356
3.1414232
3.1417984
3.1418232
piGPU               :    CPU: 65670.783 us   +/- 88.005 (min: 65595.754 / max: 65857.329) us     GPU-0: 66025.008 us   +/- 20.379 (min: 65997.475 / max: 66061.951) us


**Ejercicio**:
* Comparen la performance para distintos $n$

# curand DEVICE API

## Calcular $\pi$ usando la DEVICE API de curand

In [ ]:
%%writefile pi.cu

#include <thrust/iterator/counting_iterator.h>
#include <thrust/functional.h>
#include <thrust/transform_reduce.h>
#include <curand_kernel.h>

#include <iostream>
#include <iomanip>

#define GLOBALSEED  12345

// we could vary M & N to find the perf sweet spot

struct estimate_pi :
    public thrust::unary_function<unsigned int, float>
{
  __device__
  float operator()(unsigned int thread_id)
  {
    float sum = 0;
    unsigned int N = 10000; // samples per thread
    unsigned int seed = thread_id;

    curandStatePhilox4_32_10_t s;

    // seed a random number generator
    curand_init(GLOBALSEED,seed, 0, &s);

    // take N samples in a quarter circle
    for(unsigned int i = 0; i < N; ++i)
    {
      // draw a sample from the unit square
      float x = curand_uniform(&s);
      float y = curand_uniform(&s);

      // measure distance from the origin
      float dist2 = (x*x + y*y);

      // add 1.0f if (u0,u1) is inside the quarter circle
      if(dist2 <= 1.0f)
        sum += 1.0f;
    }

    // multiply by 4/N to get the area of the whole circle
    sum *= 4.0f/N;

    return sum;
  }
};

int main(void)
{
  // use 30K independent seeds
  int M = 30000;

  float estimate = thrust::transform_reduce(
        thrust::counting_iterator<int>(0),
        thrust::counting_iterator<int>(M),
        estimate_pi(),
        0.0f,
        thrust::plus<float>());
  estimate /= M;

  std::cout << std::setprecision(6);
  std::cout << "pi is approximately ";
  std::cout << estimate << std::endl;
  return 0;
}

//if(thread_id<=2) printf("%d %d %d\n",thread_id,s.key.x,s.ctr.x);
//if(thread_id<=2) printf("%d %d %d\n",thread_id,s.key.x,s.ctr.x);


Writing pi.cu


In [ ]:
!nvcc -arch=sm_75 -lcurand pi.cu -o pi
!./pi

pi is approximately 3.14143


## Calcular $\pi$ usando la DEVICE API de curand en python con cupy RawKernel


In [ ]:
import cupy as cp

# Raw CUDA code for generating random numbers using curand
curand_kernel_code = """
#include <curand_kernel.h>

extern "C" __global__ void generate_randoms(float *out, int n, unsigned long long seed) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;

    // Initialize curand state for each thread
    curandState state;
    curand_init(seed, idx, 0, &state);

    // Generate a random float and store it in the output array
    if (idx < n) {
        int M=10000;
        float aciertos=0;
        for(int i=0;i<M;i++)
        {
          float x = curand_uniform(&state);  // Generates random float between 0 and 1
          float y = curand_uniform(&state);  // Generates random float between 0 and 1
          if(x*x+y*y<1) aciertos++;
        }
        out[idx] = 4*aciertos/M;
    }
}
"""

# Kernel function using CuPy's RawKernel
curand_kernel = cp.RawKernel(curand_kernel_code, "generate_randoms")

# Array to store random numbers
n = 1024  # Number of random numbers to generate
out_array = cp.empty(n, dtype=cp.float32)

# Block and grid dimensions
block_size = 256
grid_size = (n + block_size - 1) // block_size

# Launch kernel to generate random numbers
curand_kernel((grid_size,), (block_size,), (out_array, n, 1234))

# Print the generated random numbers
print(out_array)

print("la estimacion de Pi es ", out_array.mean())


[3.134  3.1596 3.1524 ... 3.1176 3.1636 3.158 ]
la estimacion de Pi es  3.1416094


## Calcular $\pi$ usando la DEVICE API de numba




Numba tiene un generador que puede ser llamado desde el kernel, es decir una funcion de device. El unico generador disponible por ahora es el xoroshiro128p_states.

In [1]:
#%%writefile numbarand.py
from __future__ import print_function, absolute_import

from numba import cuda
from numba.cuda.random import create_xoroshiro128p_states, xoroshiro128p_uniform_float32
import numpy as np

@cuda.jit
def compute_pi(rng_states, iterations, out):
    """Find the maximum value in values and store in result[0]"""
    thread_id = cuda.grid(1)

    # Compute pi by drawing random (x, y) points and finding what
    # fraction lie inside a unit circle
    inside = 0
    for i in range(iterations):
        x = xoroshiro128p_uniform_float32(rng_states, thread_id)
        y = xoroshiro128p_uniform_float32(rng_states, thread_id)
        if x**2 + y**2 <= 1.0:
            inside += 1

    out[thread_id] = 4.0 * inside / iterations

threads_per_block = 64
blocks = 24
rng_states = create_xoroshiro128p_states(threads_per_block * blocks, seed=1)
out = np.zeros(threads_per_block * blocks, dtype=np.float32)

compute_pi[blocks, threads_per_block](rng_states, 10000, out)
print('pi:', out.mean())

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 24 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


pi: 3.1416733


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


# Libreria Externa Random123


Vamos a traernos la librería Random123 que es externa al cuda toolkit, aunque el toolkit la incorpore en parte, porque esta solo deja usar su implementación en GPU, pero la librería también sirve para CPU.

In [ ]:
#@ Bajar carpeta common, que tiene random123 y otras cosas utiles
!pip install --upgrade gdown
!gdown --folder --id 1lSpHInmykcakxN3N3DbEgn6F_AA6k24R -O /content/


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Retrieving folder contents
Retrieving folder 1anaIAkuxzF5dIPp_klzloXR5ft2-6CVZ Random123
Retrieving folder 1jya7IPxM8hQ0XnCna0g7zXyyFMYZYoyq conventional
Processing file 115SfbKFJsyq6IUGOakJ3x3Yle4NhLvcj Engine.hpp
Processing file 1Gx7mC91Dr_74ttk4WsLESZYsmHEh59PA gsl_cbrng.h
Retrieving folder 1oYx1-HRHCHlP90T6C1DkP94A6nI3bSYm features
Processing file 1bgqfvn4qNBQeLUXLQtnqL2bNYjRhnJrG clangfeatures.h
Processing file 1v6npSKTkRD34vNcmNdlgPAhrJ8qgLyaW compilerfeatures.h
Processing file 1eYzGk_lXZcgPZFLR8hMWZto69Xu0XcQd gccfeatures.h
Processing file 1Vnn1vVsxBnBSnYMQ8Kgb80bj4wy9c_yU iccfeatures.h
Processing file 1I_CUq5nX2Wo9A-JW-4HBVwy75jmnzjjJ msvcfeatures.h
Processing file 1Z71LUZ1ti1amA5zvtqmrM-pmAfYab0NZ nvccfeatures.h
Processing file 1ihGQuOhoJEUXlcVbMLtV3qc7PuVttqn

## Calcular $\pi$

In [38]:
%%writefile piR123.cu

#include <thrust/iterator/counting_iterator.h>
#include <thrust/functional.h>
#include <thrust/transform_reduce.h>
#include <thrust/functional.h>
#include <chrono> // Standard C++ timing
#include <Random123/philox.h>

#include <iostream>
#include <iomanip>

#ifndef __CUDACC__
#define __device__
#define __host__
#endif


#define GLOBALSEED  12345

// we could vary M & N to find the perf sweet spot


struct estimate_pi
{
    __device__
    float operator()(unsigned int thread_id)
    {
        float sum = 0;
        unsigned int N = 10000;

        // Random123 Setup
        // philox2x32 yields two 32-bit uints per call
        typedef r123::Philox2x32 G;
        G::key_type k = {{thread_id}}; // Use thread_id as the key
        G::ctr_type c = {{0}};         // Initialize counter

        // take N samples in a quarter circle
        for(unsigned int i = 0; i < N; ++i)
        {
            // Increment counter for each iteration to get new randomness
            c[0] = i;
            G::ctr_type r = G()(c, k);

            // Convert raw bits (uint32) to float in [0, 1]
            // 2.3283064e-10f is 1.0 / (2^32)
            float x = r[0] * 2.3283064e-10f;
            float y = r[1] * 2.3283064e-10f;

            if(x*x + y*y <= 1.0f)
                sum += 1.0f;
        }

        return (sum * 4.0f) / N;
    }
};

int main(void)
{
  // use 30K independent seeds
  int M = 30000;

  auto start = std::chrono::high_resolution_clock::now();

  float estimate = thrust::transform_reduce(
        thrust::counting_iterator<int>(0),
        thrust::counting_iterator<int>(M),
        estimate_pi(),
        0.0f,
        thrust::plus<float>());
  estimate /= M;

  auto end = std::chrono::high_resolution_clock::now();
  std::chrono::duration<double, std::milli> duration = end - start;

  std::cout << std::setprecision(6);
  std::cout << "pi is approximately ";
  std::cout << estimate << std::endl;
  std::cout << "Calculation took: " << duration.count() << " ms" << std::endl;
  return 0;
}


//if(thread_id<=2) printf("%d %d %d\n",thread_id,s.key.x,s.ctr.x);
//if(thread_id<=2) printf("%d %d %d\n",thread_id,s.key.x,s.ctr.x);


Overwriting piR123.cu


In [39]:
#@title CUDA BACKEND

%%bash

echo "Compilando cuda backend..."
nvcc -arch=sm_75 -DTHRUST_DEVICE_SYSTEM=THRUST_DEVICE_SYSTEM_CUDA \
piR123.cu -o piR123 -I/content/common
echo

echo "Corriendo"
./piR123


Compilando cuda backend...

Corriendo
pi is approximately 3.14172
Calculation took: 257.143 ms


Si quiero paralelizar en CPU usando OpenMP Thrust backend, no puedo usar curand, pero puedo seguir usando random123.

In [40]:
#@title OPENMP BACKEND
%%bash

cp piR123.cu piR123.cpp

echo "Compilando omp backend..."
g++ -fopenmp -lgomp piR123.cpp \
-DTHRUST_DEVICE_SYSTEM=THRUST_DEVICE_SYSTEM_OMP \
-I/usr/local/cuda-12.8/targets/x86_64-linux/include/ \
-I/content/common \
-o piR123

echo
echo "Corriendo"
./piR123

Compilando omp backend...

Corriendo
pi is approximately 3.14172
Calculation took: 31275.2 ms


In [41]:
#@title CPP BACKEND
%%bash

cp piR123.cu piR123.cpp

echo "Compilando omp backend..."
g++ -fopenmp -lgomp piR123.cpp \
-DTHRUST_DEVICE_SYSTEM=THRUST_DEVICE_SYSTEM_CPP \
-I/usr/local/cuda-12.8/targets/x86_64-linux/include/ \
-I/content/common \
-o piR123

echo
echo "Corriendo"
./piR123

Compilando omp backend...

Corriendo
pi is approximately 3.14173
Calculation took: 48869.4 ms
